# Finance headline sentiment: DPO training

Fine-tunes Qwen2.5-0.5B-Instruct with DPO (Hugging Face TRL, LoRA) so it labels financial news headlines as
bearish, bullish or neutral, answering only in JSON. Training runs on both T4 GPUs with Accelerate and DeepSpeed
ZeRO-2 and is tracked in Weights & Biases. The serving notebook scores the result with vLLM on 500 held-out headlines.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q trl==1.16.0 peft==0.21.2 accelerate==1.15.0 transformers==5.20.2 datasets deepspeed wandb
# Kaggle ships torchao 0.10, and PEFT refuses to build LoRA layers while that old version is installed.
!pip uninstall -y -q torchao
!pip list 2>/dev/null | grep -iE "^(torch|transformers|trl|peft|accelerate|deepspeed|vllm|wandb) "

In [ ]:
import os
# W&B runs offline here and is synced afterwards, so no API key is stored in the notebook.
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_PROJECT"] = "finsent-dpo"
os.environ["WANDB_DIR"] = "/kaggle/working"

In [ ]:
%%writefile finsent.py
"""Shared code for the financial-headline sentiment project.

The model must answer with JSON such as {"label": "bullish"}. A deterministic checker scores each answer,
and a word-list challenger flags the headlines a person should review, the same design as Reconcile.
"""
import json
import random
import re

LABELS = ["bearish", "bullish", "neutral"]  # dataset labels 0, 1 and 2
# Run 2 prompt. Run 1 showed the labels as {"label": "bearish" | "bullish" | "neutral"}, and the base model
# copied that pattern with a quote missing, so this version names the labels and gives one plain example.
PROMPT = ('Classify the sentiment of this financial news headline as bearish, bullish or neutral. '
          'Reply with JSON only, for example {"label": "neutral"}.\n\n')
PROMPT_RUN1 = ('Classify the sentiment of this financial news. '
               'Reply only with JSON: {"label": "bearish" | "bullish" | "neutral"}.\n\n')


def to_pair(example, index, rng=random):
    """Builds one DPO preference pair in conversational format.

    The rejected answer alternates between a wrong label in valid JSON (odd rows) and the right label
    wrapped in prose, which isn't valid JSON (even rows).
    """
    gold = LABELS[example["label"]]
    wrong = rng.choice([lab for lab in LABELS if lab != gold])
    rejected = json.dumps({"label": wrong}) if index % 2 else f"Sure! The sentiment is {gold}."
    return {
        "prompt": [{"role": "user", "content": PROMPT + example["text"]}],
        "chosen": [{"role": "assistant", "content": json.dumps({"label": gold})}],
        "rejected": [{"role": "assistant", "content": rejected}],
    }


def check(raw, gold):
    """Strict checker: the whole answer must parse as JSON with an allowed label."""
    try:
        obj = json.loads(raw.strip())
    except (json.JSONDecodeError, AttributeError):
        return {"valid_json": False, "allowed_label": False, "label": None, "correct": False}
    label = str(obj.get("label", "")).strip().lower() if isinstance(obj, dict) else ""
    allowed = label in LABELS
    return {"valid_json": True, "allowed_label": allowed, "label": label if allowed else None,
            "correct": allowed and label == gold}


BULLISH = ["beat", "beats", "raise", "raises", "raised", "upgrade", "upgrades", "upgraded", "surge", "surges",
           "soar", "soars", "jump", "jumps", "gain", "gains", "rally", "rallies", "record", "higher", "rise",
           "rises", "buy", "outperform", "bullish", "boost", "boosts", "strong", "tops", "climbs"]
BEARISH = ["miss", "misses", "missed", "cut", "cuts", "downgrade", "downgrades", "downgraded", "fall", "falls",
           "drop", "drops", "plunge", "plunges", "slump", "slumps", "tumble", "tumbles", "lower", "loss",
           "losses", "sell", "underperform", "bearish", "decline", "declines", "slide", "slides", "warns",
           "weak", "layoffs", "lawsuit"]
_WORD = re.compile(r"[a-z]+")


def rules_label(text):
    """Word-list challenger: more bullish words than bearish ones means bullish, and so on."""
    words = _WORD.findall(text.lower())
    score = sum(w in BULLISH for w in words) - sum(w in BEARISH for w in words)
    return "bullish" if score > 0 else "bearish" if score < 0 else "neutral"


def summarize(outputs, golds, texts):
    """Scores a model's raw answers. Disagreements with the challenger form the review queue."""
    rows = [check(o, g) for o, g in zip(outputs, golds)]
    n = len(rows)
    review = [r["label"] != rules_label(t) for r, t in zip(rows, texts)]
    wrong = [not r["correct"] for r in rows]
    caught = sum(w and q for w, q in zip(wrong, review))
    return {
        "n": n,
        "valid_json_rate": sum(r["valid_json"] for r in rows) / n,
        "accuracy": sum(r["correct"] for r in rows) / n,
        "review_rate": sum(review) / n,
        "errors": sum(wrong),
        "errors_in_review": caught,
    }

In [ ]:
import random
from datasets import load_dataset
from finsent import to_pair

random.seed(0)
ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
train = ds["train"].shuffle(seed=42).select(range(2000))
pairs = train.map(to_pair, with_indices=True, remove_columns=train.column_names,
                  load_from_cache_file=False)  # the cache would keep pairs built with an older prompt
pairs.save_to_disk("/kaggle/working/dpo_train")
print(pairs[0])
print(pairs[1])

In [ ]:
%%writefile train_dpo.py
"""DPO fine-tuning of Qwen2.5-0.5B-Instruct with LoRA, launched by Accelerate (DeepSpeed ZeRO-2 on Kaggle).

Set SMOKE=1 for a two-step CPU run that only checks the code path.
"""
import json
import os

from datasets import load_from_disk
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_DIR = os.environ.get("DATA_DIR", "/kaggle/working/dpo_train")
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/qwen-dpo")
SMOKE = os.environ.get("SMOKE") == "1"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)  # fp32 master weights; fp16 mixed precision on the T4s
train = load_from_disk(DATA_DIR)
if SMOKE:
    train = train.select(range(2))

args = DPOConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1 if SMOKE else 4,
    gradient_accumulation_steps=1 if SMOKE else 4,
    learning_rate=5e-5,
    num_train_epochs=1,
    beta=0.1,
    # Run 1 used DPO alone, and the chosen answers' likelihood fell along with the rejected ones. The added SFT
    # term trains on the chosen answers directly, so the right label in valid JSON is rewarded as well as preferred.
    loss_type=["sigmoid", "sft"],
    loss_weights=[1.0, 1.0],
    max_length=256,
    bf16=False,  # TRL turns bf16 on by default, but T4 GPUs don't support it
    fp16=not SMOKE,
    use_cpu=SMOKE,
    logging_steps=5,
    save_strategy="no",
    report_to="none" if SMOKE else "wandb",
    run_name="qwen0.5b-dpo-sft-finsent",
    seed=42,
    max_steps=2 if SMOKE else -1,
)
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
# With a LoRA adapter, TRL uses the frozen base model as the reference, so no second copy is loaded.
trainer = DPOTrainer(model=model, ref_model=None, args=args, train_dataset=train,
                     processing_class=tok, peft_config=peft_config)
trainer.train()
trainer.save_model(OUT_DIR)
if trainer.accelerator.is_main_process:
    with open(os.path.join(OUT_DIR, "log_history.json"), "w") as fh:
        json.dump(trainer.state.log_history, fh, indent=1)
    with open(os.path.join(OUT_DIR, "run_info.json"), "w") as fh:
        json.dump({"num_processes": trainer.accelerator.num_processes,
                   "distributed_type": str(trainer.accelerator.distributed_type),
                   "train_rows": len(train), "global_step": trainer.state.global_step}, fh, indent=1)

In [ ]:
%%writefile ds_zero2.yaml
compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  zero_stage: 2
  gradient_accumulation_steps: 4
  offload_optimizer_device: none
  offload_param_device: none
mixed_precision: fp16
num_machines: 1
num_processes: 2
machine_rank: 0
use_cpu: false

In [ ]:
import subprocess

def stream(cmd):
    """Runs a command and prints its output in this cell as it arrives. Returns the exit code."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

import json

launchers = [
    ("DeepSpeed ZeRO-2 on 2 GPUs", ["accelerate", "launch", "--config_file", "ds_zero2.yaml", "train_dpo.py"]),
    ("Accelerate multi-GPU (DDP) on 2 GPUs",
     ["accelerate", "launch", "--multi_gpu", "--num_processes", "2", "--mixed_precision", "fp16", "train_dpo.py"]),
]
for label, cmd in launchers:
    print(f"=== {label}")
    if stream(cmd) == 0:
        json.dump({"launcher": label}, open("/kaggle/working/launcher.json", "w"))
        break
else:
    raise RuntimeError("Training failed with both launchers")

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

merged = AutoPeftModelForCausalLM.from_pretrained("/kaggle/working/qwen-dpo", dtype=torch.float16).merge_and_unload()
merged.save_pretrained("/kaggle/working/qwen-dpo-merged")
AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct").save_pretrained("/kaggle/working/qwen-dpo-merged")
print(open("/kaggle/working/launcher.json").read())
print(open("/kaggle/working/qwen-dpo/run_info.json").read())